In [ ]:
%pip install tensorflow nltk scikit-learn pandas numpy
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

In [ ]:
import numpy as np
import pandas as pd
import nltk
import re
import string

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
import os
import json

os.makedirs('/root/.kaggle', exist_ok=True)

token_data = {
    "username": "Enter username",
    "key": "Enter generated api key"
}

with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(token_data, f)

os.chmod('/root/.kaggle/kaggle.json', 600)
print("Kaggle configured successfully!")

In [ ]:
!pip install kaggle -q
!kaggle datasets download -d yelp-dataset/yelp-dataset
!unzip -q yelp-dataset.zip
!ls

In [ ]:
from io import StringIO

chunks = []
with open('yelp_academic_dataset_review.json', 'r') as f:
    for i, line in enumerate(f):
        if i >= 50000:  # Read more lines to get better balance
            break
        chunks.append(pd.read_json(StringIO(line), typ='series'))

df = pd.DataFrame(chunks)[['text', 'stars']].rename(columns={'text': 'review', 'stars': 'rating'})

# Map to 3 classes
def map_rating(rating):
    if rating <= 2:
        return 1
    elif rating == 3:
        return 3
    else:
        return 5

df['rating'] = df['rating'].apply(map_rating)

# Balance classes
neg = df[df['rating'] == 1]
neu = df[df['rating'] == 3]
pos = df[df['rating'] == 5]

min_count = min(len(neg), len(neu), len(pos))
print(f"Samples per class: {min_count}")

df = pd.concat([
    neg.sample(min_count, random_state=42),
    neu.sample(min_count, random_state=42),
    pos.sample(min_count, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(df.shape)
print(df['rating'].value_counts())
print(df.head())

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Lowercase
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stop words
    tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(tokens)

df['cleaned_review'] = df['review'].apply(clean_text)
print("\nOriginal vs Cleaned:")
print(df[['review', 'cleaned_review']].head(3))

In [ ]:
# Config
MAX_WORDS = 10000   # Max vocabulary size
MAX_LEN = 50        # Max sequence length

# Tokenize text
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(df['cleaned_review'])

sequences = tokenizer.texts_to_sequences(df['cleaned_review'])
print(f"\nVocabulary size: {len(tokenizer.word_index)}")
print(f"Sample sequence: {sequences[0]}")

In [ ]:
padded_sequences = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')
print(f"\nShape after padding: {padded_sequences.shape}")
print(f"Sample padded sequence:\n{padded_sequences[0]}")

In [ ]:
# Map ratings to classes: 1-2 = negative, 3 = neutral, 4-5 = positive
def map_rating(rating):
    if rating <= 2:
        return 0  # Negative
    elif rating == 3:
        return 1  # Neutral
    else:
        return 2  # Positive

df['label'] = df['rating'].apply(map_rating)

labels = np.array(df['label'])
print(f"\nLabel distribution: {np.bincount(labels)}")
print("0=Negative, 1=Neutral, 2=Positive")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, labels,
    test_size=0.2,
    random_state=42,
    stratify=labels if len(np.unique(labels)) > 1 else None
)

print(f"\nTraining samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")

In [ ]:
EMBEDDING_DIM = 64
NUM_CLASSES = 3

model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(LSTM(32)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=4,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Loss:     {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# Detailed predictions
predictions = model.predict(X_test)
predicted_classes = np.argmax(predictions, axis=1)

label_names = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

print("\n--- Prediction Results ---")
for i in range(len(X_test)):
    actual = label_names[y_test[i]]
    predicted = label_names[predicted_classes[i]]
    confidence = predictions[i][predicted_classes[i]] * 100
    status = "✓" if actual == predicted else "✗"
    print(f"{status} Actual: {actual:9s} | Predicted: {predicted:9s} | Confidence: {confidence:.1f}%")

In [ ]:
# Step 1: Rebuild cleaned column
df['cleaned_review'] = df['review'].apply(clean_text)
print("Cleaned column created:")
print(df['cleaned_review'].head())

# Step 2: Rebuild label column
def map_rating(rating):
    if rating <= 2:
        return 0  # Negative
    elif rating == 3:
        return 1  # Neutral
    else:
        return 2  # Positive

df['label'] = df['rating'].apply(map_rating)

# Step 3: Rebuild tokenizer
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(df['cleaned_review'])

# Step 4: Rebuild sequences and padding
sequences = tokenizer.texts_to_sequences(df['cleaned_review'])
padded_sequences = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

# Step 5: Verify tokenizer
test_seq = tokenizer.texts_to_sequences(['worst product ever bought'])
print(f"\nTest sequence: {test_seq}")  # Should show integers not None

# Step 6: Predict function
def predict_review(review_text):
    cleaned = clean_text(review_text)
    sequence = tokenizer.texts_to_sequences([cleaned])
    sequence = [[0 if token is None else token for token in seq] for seq in sequence]
    padded = pad_sequences(sequence, maxlen=MAX_LEN, padding='post', truncating='post')
    prediction = model.predict(padded, verbose=0)
    predicted_class = int(np.argmax(prediction[0]))
    confidence = float(prediction[0][predicted_class]) * 100
    return label_names[predicted_class], confidence

# Step 7: Test
new_reviews = [
    "This is the worst product I have ever bought!",
    "Really happy with my purchase, great quality!",
    "It is decent, nothing too great or too bad."
]

print("\n--- New Review Predictions ---")
for review in new_reviews:
    sentiment, confidence = predict_review(review)
    print(f"Review:    '{review}'")
    print(f"Sentiment: {sentiment} ({confidence:.1f}% confidence)\n")

In [ ]:
# Test with tricky reviews
tricky_reviews = [
    "Not bad at all, quite enjoyed it",           # Double negative = positive
    "Could have been worse I suppose",             # Backhanded neutral
    "It works, I guess",                           # Unenthusiastic neutral
    "Surprisingly didn't break immediately",       # Sarcastic
    "My cat loves sitting on the box it came in",  # Irrelevant
]

print("\n--- Tricky Review Predictions ---")
for review in tricky_reviews:
    sentiment, confidence = predict_review(review)
    print(f"Review:    '{review}'")
    print(f"Sentiment: {sentiment} ({confidence:.1f}% confidence)\n")

real_reviews = [
    "Absolutely loved this place, food was incredible!",
    "Worst experience ever, never coming back.",
    "It was okay, nothing special.",
    "Amazing service and great atmosphere!",
    "Very disappointing, expected much better.",
]

print("\n--- Real Review Predictions ---")
for review in real_reviews:
    sentiment, confidence = predict_review(review)
    print(f"Review:    '{review}'")
    print(f"Sentiment: {sentiment} ({confidence:.1f}% confidence)\n")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.legend()

plt.tight_layout()
plt.savefig('training_curves.png')
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

y_pred = np.argmax(model.predict(X_test), axis=1)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Neutral', 'Positive'],
            yticklabels=['Negative', 'Neutral', 'Positive'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.savefig('confusion_matrix.png')
plt.show()

print(classification_report(y_test, y_pred, 
      target_names=['Negative', 'Neutral', 'Positive']))

In [ ]:
plt.figure(figsize=(6, 4))
df['rating'].value_counts().sort_index().plot(kind='bar', color=['red', 'gray', 'green'])
plt.title('Dataset Distribution')
plt.xlabel('Sentiment')
plt.xticks([0, 1, 2], ['Negative', 'Neutral', 'Positive'], rotation=0)
plt.ylabel('Count')
plt.savefig('distribution.png')
plt.show()